# 11 -- Проверки

Проверяю формулу метрики, формат CSV и границы разбиений. Тесты не используют скрытую разметку.
Seed 143. Для нового файла ответы проверяю тем же контрактом, что и для исходного.

In [1]:
from pathlib import Path

ck_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
get_ipython().run_line_magic('run', str(ck_root / 'notebooks/common.ipynb'))


In [2]:
import tempfile
import unittest
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse


class Contracts(unittest.TestCase):
    def test_macro_recall_counts_missing_queries(self):
        truth = {'a': ['x', 'y'], 'b': ['z'], 'c': ['v']}
        result = recall_table({'a': ['x'], 'b': ['z']}, truth, ks=(50,))
        self.assertAlmostEqual(result['recall@50'].mean(), 0.5)
        with self.assertRaises(ValueError):
            recall_table({'a': ['x', 'x']}, {'a': ['x']})

    def test_deterministic_ties_and_local_top(self):
        scores = np.array([1.0, 3.0, 3.0, 0.0, 2.0])
        self.assertEqual(top_indices(scores, 2).tolist(), [1, 2])
        self.assertEqual(top_indices(scores, 2, [0, 3, 4]).tolist(), [4, 0])
        idx, val = sparse_top(
            sparse.csr_matrix(scores.reshape(1, -1)), 2, np.array([1, 0, 0, 1, 1], bool)
        )
        self.assertEqual(idx.tolist(), [4, 0])
        self.assertEqual(top_indices(np.zeros(4)).tolist(), [])

    def test_bm25_matches_hand_formula(self):
        counts = sparse.csr_matrix(
            [[2.0, 0.0], [1.0, 1.0], [0.0, 3.0]], dtype=np.float32
        )
        got = LexicalIndex.bm25_matrix(counts).toarray()
        lengths = np.array([2.0, 2.0, 3.0])
        avg = lengths.mean()
        expected = (
            2
            * 2.2
            / (2 + 1.2 * (0.25 + 0.75 * lengths[0] / avg))
            * np.log1p((3 - 2 + 0.5) / (2 + 0.5))
        )
        self.assertAlmostEqual(float(got[0, 0]), float(expected), places=6)
        self.assertEqual(got[0, 1], 0)

    def test_rrf_deduplicates_items(self):
        c = pd.DataFrame(
            [['q', 0, 'a', 1, 1.0], ['q', 0, 'b', 2, 1.0], ['q', 1, 'a', 2, 0.5]],
            columns=['query_id', 'pos', 'source', 'rank', 'score'],
        )
        result = rrf(c, {'a': 1, 'b': 1}, top_k=2)
        self.assertEqual(result.pos.tolist(), [0, 1])
        self.assertEqual(len(result), 2)

    def test_normalization_and_context(self):
        self.assertEqual(normalize('  Ёлка\nРЕМОНТ  12 '), 'елка ремонт 12')
        q = pd.DataFrame(
            {
                'search_query': ['Ёлка', 'елка'],
                'search_location_id': [1, 2],
                'search_is_delivery_search': [0, 0],
                'search_infm_params_text': ['', ''],
                'search_category': [114, 114],
            }
        )
        p = prepare_queries(q)
        self.assertEqual(p.text_key.nunique(), 1)
        self.assertEqual(p.context_key.nunique(), 2)

    def test_price_sentinel_is_not_free(self):

        frame = pd.DataFrame(
            {
                'item_price': [-1.0, 0.0, 100.0],
                'item_rating_reviews_count': [0.0, 1.0, 10.0],
                'item_rating': [float('nan'), 5.0, 4.0],
                'title_text': ['a'] * 3,
                'params_text': [''] * 3,
                'description_text': [''] * 3,
                'item_is_phone_hidden': [False] * 3,
                'item_is_message_forbidden': [False] * 3,
                'item_microcat_id': [1] * 3,
                'item_latitude': [55.0] * 3,
                'item_longitude': [37.0] * 3,
            }
        )
        features = item_features(frame)
        self.assertEqual(features.item_price_unknown.tolist(), [1.0, 0.0, 0.0])
        self.assertTrue(pd.isna(features.loc[0, 'item_price_log1p']))
        self.assertEqual(features.loc[1, 'item_price_log1p'], 0.0)
        self.assertTrue(pd.isna(features.loc[0, 'price_vs_microcat_log']))

    def test_inference_absent_channel_has_zero_rank(self):

        frame = pd.DataFrame({'rating': [float('nan')], 'rank_word': [0.1]})
        result = model_matrix(frame, ['rating', 'rank_word', 'rank_dense'])
        self.assertEqual(result.loc[0, 'rating'], -1.0)
        self.assertEqual(result.loc[0, 'rank_dense'], 0.0)

    def test_csv_contract_and_leading_zero(self):
        q = ['00WuFMaXSFZBxSzT']
        ids = ['0123456789abcdef', '1111111111111111']
        with tempfile.TemporaryDirectory() as directory:
            p = Path(directory) / 'answer.csv'
            p.write_text(
                'query_id,answer\n' + q[0] + ',' + ids[0] + '\n', encoding='utf-8'
            )
            self.assertTrue(validate_submission(p, q, ids)['valid'])
            for bad in [
                ids[0] + ' ' + ids[0],
                ids[0].upper(),
                ids[0] + '  ' + ids[1],
                'not_an_item_id___',
            ]:
                p.write_text(
                    'query_id,answer\n' + q[0] + ',' + bad + '\n', encoding='utf-8'
                )
                with self.assertRaises(ValueError):
                    validate_submission(p, q, ids)
            p.write_text('query_id,answer\n', encoding='utf-8')
            with self.assertRaises(ValueError):
                validate_submission(p, q, ids)
            p.write_text('query_id,answer\n' + q[0] + ',\n', encoding='utf-8')
            self.assertEqual(validate_submission(p, q, ids)['min_candidates'], 0)


In [3]:
ck_suite = unittest.defaultTestLoader.loadTestsFromTestCase(Contracts)
ck_result = unittest.TextTestRunner(verbosity=2).run(ck_suite)
assert ck_result.wasSuccessful()


test_bm25_matches_hand_formula (__main__.Contracts.test_bm25_matches_hand_formula) ... 

ok


test_csv_contract_and_leading_zero (__main__.Contracts.test_csv_contract_and_leading_zero) ... 

ok


test_deterministic_ties_and_local_top (__main__.Contracts.test_deterministic_ties_and_local_top) ... 

...

test_price_sentinel_is_not_free (__main__.Contracts.test_price_sentinel_is_not_free) ... 

ok


test_rrf_deduplicates_items (__main__.Contracts.test_rrf_deduplicates_items) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 0.016s

OK


## Разбиения и кандидаты

Проверяю полные контексты, cold-тексты, соответствие truth и индексы объявлений.

In [4]:
"""Проверяет связь разбиений, кандидатов и индексов после прогона."""

import json
from pathlib import Path
import pandas as pd

root = ck_root
cache = root / 'outputs/cache'
pairs = pd.read_parquet(
    cache / 'split_pairs.parquet',
    columns=['context_key', 'text_key', 'item_id', 'split', 'regime', 'query_id'],
)
eval_q = pd.read_parquet(cache / 'eval_queries.parquet')
base = pairs[pairs.split.eq('base')]
held = pairs[pairs.split.ne('base')]
assert not set(base.context_key) & set(held.context_key)
assert not set(base.text_key) & set(held.loc[held.regime.eq('cold'), 'text_key'])
assert eval_q.query_id.is_unique
truth = json.loads((cache / 'truth.json').read_text())
expected = (
    held[held.query_id.isin(eval_q.query_id)]
    .groupby('query_id')
    .item_id.agg(lambda s: sorted(set(s)))
    .to_dict()
)
assert truth == expected
items = pd.read_parquet(cache / 'items.parquet', columns=['item_id'])
assert items.item_id.is_unique
allowed = set(items.item_id)
assert all(set(v) <= allowed for v in truth.values())
for a, b in [('rank_train', 'dev'), ('rank_train', 'test'), ('dev', 'test')]:
    assert not set(eval_q.loc[eval_q.split.eq(a), 'text_key']) & set(
        eval_q.loc[eval_q.split.eq(b), 'text_key']
    )
summary = {}
for path in sorted(cache.glob('candidates_*.parquet')):
    c = pd.read_parquet(path)
    assert set(c.query_id) <= set(eval_q.query_id), path.name
    assert c.pos.between(0, len(items) - 1).all(), path.name
    assert not c.duplicated(['query_id', 'source', 'pos']).any(), path.name
    assert c['rank'].gt(0).all(), path.name
    summary[path.name] = {
        'rows': len(c),
        'queries': c.query_id.nunique(),
        'sources': c.source.nunique(),
    }
(root / 'outputs/validation/artifact_audit.json').write_text(
    json.dumps(
        {
            'valid': True,
            'queries': len(eval_q),
            'items': len(items),
            'candidate_files': summary,
        },
        indent=2,
    )
)
print('Split, truth and candidate artifact invariants: OK')


Split, truth and candidate artifact invariants: OK


## Файлы для отправки

In [5]:
ck_queries = pd.read_parquet(
    data_dir() / 'benchmark_queries.parquet', columns=['query_id']
)
ck_items = pd.read_parquet(data_dir() / 'benchmark_items.parquet', columns=['item_id'])
for ck_path in sorted((ck_root / 'outputs/submissions').glob('answer*.csv')):
    print(
        ck_path.name,
        validate_submission(
            ck_path, ck_queries.query_id.tolist(), ck_items.item_id.tolist()
        ),
    )


answer.csv {'rows': 2452, 'min_candidates': 50, 'max_candidates': 50, 'valid': True}


answer_blend005.csv {'rows': 2452, 'min_candidates': 50, 'max_candidates': 50, 'valid': True}


answer_extended.csv {'rows': 2452, 'min_candidates': 50, 'max_candidates': 50, 'valid': True}


## Стабильность признаков при новом канале

В локальном split нет history_context, а на benchmark он может появиться.
Фиксирую список каналов для новых признаков по обучению. Проверяю, что появление лишней колонки
не меняет знаменатель retriever_agreement. Это реальная найденная проблема согласования train/inference,
а не изменение формулы Recall.

In [6]:
ck_frame = (
    pd.read_parquet(ck_root / 'outputs/cache/fusion_dev.parquet').iloc[:100].copy()
)
ck_catalog = pd.read_parquet(ck_root / 'outputs/cache/items.parquet')
ck_eval_queries = pd.read_parquet(ck_root / 'outputs/cache/eval_queries.parquet')
ck_idf = json.loads((ck_root / 'outputs/models/title_idf.json').read_text())
ck_rank_columns = [col for col in ck_frame if col.startswith('rank_')]
ck_before = extra_features(
    ck_frame, ck_eval_queries, ck_catalog, ck_idf, rank_columns=ck_rank_columns
)
ck_after = extra_features(
    ck_frame.assign(rank_history_context=1 / 31),
    ck_eval_queries,
    ck_catalog,
    ck_idf,
    rank_columns=ck_rank_columns,
)
np.testing.assert_array_equal(
    ck_before.retriever_agreement, ck_after.retriever_agreement
)
print('Fixed retrieval schema: OK')


Fixed retrieval schema: OK
